# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoders:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)
- Decoders:
    - Qwen3-0.6B (Qwen/Qwen3-0.6B) (0.6B parameters)
    - Llama-3.2-1B (meta-llama/Llama-3.2-1B) (1B parameters)

In [1]:
!pip install mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 19.3 MB/s eta 0:00:00


In [1]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets.dataset_dict import DatasetDict
from datasets.arrow_dataset import Dataset
from datasets import load_dataset

import datasets
import os

In [2]:
import mteb

In [4]:
tasks = mteb.get_tasks(
    languages=["eng"],
    exclude_aggregate=True,
    task_types=[
        "Classification",
        "Retrieval",
        "Summarization",
    ],
    modalities=["text"]
)

In [ ]:
datasets.utils.logging.disable_progress_bar()
os.environ["HF_DATASETS_DISABLE_CACHE"] = "1"


task_texts_length = {}
tasks_names = {}

for task in tasks:

    if task.languages != ["eng"]:
        continue

    task.load_data()
    dataset = task.dataset
    texts_len = []

    while not isinstance(dataset, Dataset):
        dataset = list(dataset.values())[0]

    column = "text"
    if not column in dataset.features:
        column = "sentences"

    for text in dataset[column]:
        texts_len.append(len(text))
    tasks_names[task.metadata.name] = task
    task_texts_length[task.metadata.name] = {
        "min_len": min(texts_len),
        "max_len": max(texts_len),
        "mean_len": np.mean(texts_len)
    }

## Baseline models evaluation

In [89]:
xml_roberta_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

def chunk_text(text, chunk_size=512, overlap=62):
    token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    chunks = []
    start = 0
    while start < len(token_ids):
        end = start + chunk_size
        chunk_tokens = token_ids[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start += chunk_size - overlap

    return chunks

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [73]:
from transformers import AutoTokenizer, AutoModel
from langchain_text_splitters import TokenTextSplitter

"""Classes used to implement an encode funciton
passed to mteb.evaluate() funciton to evaluate on tasks
"""


def get_text_splitter(tokenizer, chunk_size=512, overlap=64):
    text_splitter = (
        TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
        .from_huggingface_tokenizer(tokenizer)
    )
    return text_splitter
    
    
class Quen3Embedding:

    name = "Qwen3-Embedding-0.6B"

    def __init__(self, chunk_size=512, overlap=64):
        self.chunk_size = chunk_size
        self.chunk_overlap = overlap
        
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
        self.text_splitter = get_text_splitter(self.tokenizer, chunk_size, overlap)

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:
        """Encodes the given sentences using the encoder.

        Args:
            inputs: The inputs to encode.
            task_metadata: The name of the task.
            hf_subset: The subset of the dataset.
            hf_split: The split of the dataset.
            prompt_type: The prompt type to use.
            **kwargs: Additional arguments to pass to the encoder.

        Returns:
            The encoded sentences.
        """

        chunks = [chunk_text(text, self.chunk_size, self.overlap) for text in inputs]
        chunks_tokenized = []
        for chunk in chunks:
            
        for chunk in chunk_tokenized:
            ouputs = self.model(chunk)
            embeddings.append(outputs.last_hidden_state)

        final_embedding = embeddings.average()
        return final_embeddin


class XMLRoBERTa:

    name = "xlm-roberta-large"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large")
        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:

        embeddings = []
        
        # for each text in input make a chunking
        texts_chunked = [chunk_text(text) for text in inputs]
        texts_chunked_tokenized = []

        for text_chunked in texts_chunked:    # for each text in chunked input texts
            # perform a tokeniztaion of each chunk
            chunk_input_ids = []
            chunk_attention_masks = []
            for chunk in text_chunked:
                tokenized = self.tokenizer(
                    chunk,
                    padding="max_length",
                    truncation=True,
                    max_length=self.chunk_size,
                    return_tensors="pt"
                )

            # concatenate all chunks in one batch
            input_ids = torch.cat(chunk_input_ids, dim=0)
            attention_mask = torch.cat(chunk_attention_masks, dim=0)

            # get embeddings for all chunks
            with torch.no_grad():
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                # mean pooling across all chunks
                embedding = outputs.last_hidden_state.mean(dim=1) 
                embeddings.append(embedding)

        return_embed = torch.cat(embeddings, dim=0)
        return return_embed


class Qwen3:

    name = "Qwen3-0.6B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("Qwen/Qwen3-0.6B")
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:
        embeddings = []

        chunk_tokenized = chunk_text(inputs, self.tokenizer, self.chunk_size)
        for chunk in chunk_tokenized:
            ouputs = self.model(chunk)
            embeddings.append(outputs.last_hidden_state)

        final_embedding = embeddings.average()
        return final_embedding

class Llama3_2:

    name = "Llama-3.2-1B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B")
        self.tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:
        embeddings = []

        chunk_tokenized = chunk_text(inputs, self.tokenizer, self.chunk_size)
        for chunk in chunk_tokenized:
            ouputs = self.model(chunk)
            embeddings.append(outputs.last_hidden_state)

        final_embedding = embeddings.average()
        return final_embedding

### Testing on example data

In [4]:
data = load_dataset("dwzhu/LongEmbed", 'qmsum')

In [5]:
test_text = data["corpus"]["text"][0]
test_text[:1000]

"Project Manager: Can I close this ?\nUser Interface: Uh we don't have any changes , do we ?\nProject Manager: Oh , okay .\nUser Interface: So no . {vocalsound}\nProject Manager: {vocalsound} There we go . Okay , here we are again . Detailed design {disfmarker} oh , come on . Well {disfmarker} Ah {gap} s Forgot to insert the minutes , but it's about the same thing we discussed before . Uh {disfmarker} Could open that anyway , think . Other design {disfmarker} anyway , we took as {disfmarker} we took w we took rubber as as the material last time . We also {gap} that you're just busy with it . Took the advanced chip to t uh implement the advanced features . Well , we discussed the design , no sharp corners , we rounded it off , like you see on the {gap} other screen , which is fine . Um {gap} we agreed that the colour should be b uh yellow and black . Yellow in the back because it's m trendy , more trendy than black anyway . So {vocalsound} then we ca yeah . We agreed that we would imple

In [74]:
# Quen3Embedding

q3_embed = Quen3Embedding(512)

In [75]:
tokenizer=q3_embed.tokenizer

In [105]:
model = q3_embed.model

In [ ]:
tok = tokenizer(test_text, 
                padding="max_length",
                truncation=True,
                return_tensors="pt")
out = model(**tok)
out.last_hidden_state

In [76]:
text_splitter = q3_embed.text_splitter

In [103]:
texts_chunked = [chunk_text(text) for text in [test_text]]
texts_chunked_tokenized = []
for text_chunked in texts_chunked:
   texts_chunked_tokenized.append(
       [tokenizer(c) for c in text_chunked]
   )

In [104]:
texts_chunked_tokenized

[[{'input_ids': [7849, 10567, 25, 2980, 358, 3265, 419, 17607, 1474, 20019, 25, 68049, 582, 1513, 944, 614, 894, 4344, 1154, 653, 582, 17607, 7849, 10567, 25, 8670, 1154, 16910, 16448, 1474, 20019, 25, 2055, 902, 659, 314, 85, 509, 1127, 795, 532, 7849, 10567, 25, 314, 85, 509, 1127, 795, 92, 2619, 582, 728, 659, 35439, 1154, 1588, 582, 525, 1549, 659, 62665, 2884, 314, 4243, 69, 27742, 92, 14019, 1154, 2525, 389, 659, 8325, 314, 4243, 69, 27742, 92, 16366, 314, 41410, 92, 274, 66174, 311, 5656, 279, 4420, 1154, 714, 432, 594, 911, 279, 1852, 3166, 582, 14078, 1573, 659, 68049, 314, 4243, 69, 27742, 92, 16503, 1787, 429, 13657, 1154, 1744, 659, 6944, 2884, 314, 4243, 69, 27742, 92, 13657, 1154, 582, 3867, 438, 314, 4243, 69, 27742, 92, 582, 3867, 289, 582, 3867, 22674, 438, 438, 279, 3684, 1537, 882, 659, 1205, 1083, 314, 41410, 92, 429, 498, 2299, 1101, 13028, 448, 432, 659, 89696, 279, 10847, 16392, 311, 259, 43744, 4211, 279, 10847, 4419, 659, 8325, 1154, 582, 14078, 279, 2884, 1154